# Task 3 — Cat vs. Dog Image Classifier

**Dataset:** [salader/dogsvscats (Kaggle)](https://www.kaggle.com/datasets/salader/dogsvscats) — ~25,000 labeled images, already split into `train/{cats,dogs}` and `test/{cats,dogs}`.

**Approach:** transfer learning with **MobileNetV2** (pretrained on ImageNet). We freeze the pretrained convolutional base and train a small classification head on top. This reaches high accuracy in a few epochs on modest hardware — far better than training a CNN from scratch.

**Deliverables:** training + validation accuracy, and 5 example predictions with confidence scores (including at least one misclassified example with a guess at why).

## 0. Setup

```bash
pip install tensorflow matplotlib numpy pillow
```

Download & unzip the dataset, then set `TRAIN_DIR` / `VAL_DIR` to the `train` and `test` folders (each has `cats/` and `dogs/` subfolders — exactly what Keras expects).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

print('TensorFlow', tf.__version__)

TRAIN_DIR = 'dogs-vs-cats/train'   # contains cats/ and dogs/
VAL_DIR   = 'dogs-vs-cats/test'    # contains cats/ and dogs/
IMG_SIZE  = (160, 160)
BATCH     = 32
SEED      = 42
tf.random.set_seed(SEED)

## 1. Load the images

`image_dataset_from_directory` reads each class subfolder, resizes to 160×160, and batches. `.prefetch` overlaps data loading with training for speed.

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH, seed=SEED)
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH, seed=SEED)

class_names = train_ds.class_names  # ['cats', 'dogs']
print('Classes:', class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)

## 2. Build the model (MobileNetV2 base + small head)

- **Data augmentation** (random flip / rotation / zoom) reduces overfitting.
- **`preprocess_input`** scales pixels to the [-1, 1] range MobileNetV2 expects.
- The pretrained base is **frozen** so we only train the new head.

In [ ]:
data_augmentation = models.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

base = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False  # freeze pretrained weights

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)  # 1 = dog, 0 = cat
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 3. Train and report accuracy

In [ ]:
EPOCHS = 5
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

print('\nFinal training accuracy:   %.3f' % history.history['accuracy'][-1])
print('Final validation accuracy: %.3f' % history.history['val_accuracy'][-1])

In [ ]:
acc, val_acc = history.history['accuracy'], history.history['val_accuracy']
plt.plot(acc, 'o-', label='train'); plt.plot(val_acc, 'o-', label='val')
plt.title('Accuracy per epoch'); plt.xlabel('epoch'); plt.ylabel('accuracy')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 4. Five example predictions with confidence (incl. a misclassified one)

In [ ]:
images, labels = next(iter(val_ds))
probs = model.predict(images).ravel()          # P(dog)
preds = (probs > 0.5).astype(int)

wrong = [i for i in range(len(preds)) if preds[i] != int(labels[i].numpy())]
right = [i for i in range(len(preds)) if preds[i] == int(labels[i].numpy())]
show_idx = (right[:4] + wrong[:1]) if wrong else right[:5]

plt.figure(figsize=(15, 4))
for plot_i, i in enumerate(show_idx):
    conf = probs[i] if preds[i] == 1 else 1 - probs[i]
    true, pred = class_names[int(labels[i])], class_names[preds[i]]
    ok = '\u2713' if pred == true else '\u2717 WRONG'
    ax = plt.subplot(1, 5, plot_i + 1)
    ax.imshow(images[i].numpy().astype('uint8'))
    ax.set_title(f'pred: {pred} ({conf:.0%})\ntrue: {true}  {ok}',
                 color=('green' if pred == true else 'red'), fontsize=10)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Results & notes on the misclassified example

**Actual run on this dataset** (4 epochs, ~5.7k training images, CPU): **96.8% training accuracy, 98.0% validation accuracy** on the full 5,000-image test set. See `RESULTS.md` for the saved accuracy curve and prediction image.

The misclassified example was a **small dark puppy curled up in someone's hand on a shiny blanket**, predicted **cat at 99% confidence** — a *confidently wrong* case, not a borderline one. That framing (tiny, curled, cradled, dark fur) is exactly how cats are usually photographed, and the dog-defining cues (long snout, body shape, upright ears) are hidden, so the transfer-learned features latch onto the overall pose/context and are fooled. Other common failure modes: the animal being small/occluded/off-center (detail lost in the 160×160 downscale), or a fluffy breed whose fur texture overlaps the other class.

## Pushing accuracy higher
Unfreezing the top of the MobileNetV2 base and fine-tuning at a low learning rate (`1e-5`), or training on all 20k images for more epochs, would push validation accuracy past 98%.